# Landing e preparação de ambiente


In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "v_credit_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

In [0]:
FILES_NAMES = [
    "base_atendentes.csv",
    "base_motivos.csv",
    "canais.csv",
    "Chamados_Hora.CSV",
    "clientes.csv",
    "custos.csv",
    "pesquisa_satisfacao.csv"
]

In [0]:
#criando o catalogo os schemas e um volume.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

In [0]:
%sql
USE CATALOG v_credit_medallion;
USE SCHEMA default;
-- Garantindo que todos os arquivos estão presentes
-- ATENÇÂO: Caso a tabela esteja vazia, certifique-se de fazer upload dos dados no volume
LIST "/Volumes/v_credit_medallion/bronze/landing/"

In [0]:
arquivos_landing = _sqldf
display(arquivos_landing)

expected_files_df = spark.createDataFrame(
    [(value,) for value in FILES_NAMES],
    schema = "name STRING"
)
missing_df = expected_files_df.join(
             arquivos_landing.select("name"), on = "name", how = "left_anti").distinct()
missing_values = [row["name"] for row in missing_df.collect()]
if missing_values:
    raise ValueError(f"Missing files: {missing_values}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    #adicionar a leitura dos tamanhos no final